In [3]:
import pandas as pd
import altair as alt
from pyarrow import lib

In [5]:
## Reading Gameweek Data

dfs = []

for gw in range (1, 39):
    df = pd.read_csv(f"data/2025-2026/gameweeks/gw{gw}.csv")
    df["gameweek"] = gw
    dfs.append(df)

gameweek_data = pd.concat(dfs, ignore_index=True)

## gameweek_data.head()

In [6]:
## Reading in Other Datasets

player_ids = pd.read_csv("data/2025-2026/player_idlist.csv")
fixtures = pd.read_csv("data/2025-2026/fixtures.csv")
teams = pd.read_csv("data/2025-2026/teams.csv")
player_data = pd.read_csv("data/2025-2026/cleaned_players.csv")

## player_ids.head()
## fixtures.head()
## teams.head()
## player_data.head()



In [7]:
## Joining player data with player_id

player_data['name'] = player_data['first_name'] + ' ' + player_data['second_name']

player_data.insert(0, 'name', player_data.pop('name'))

player_data = player_data.drop(columns=['first_name', 'second_name'])


player_ids['name'] = player_ids['first_name'] + ' ' + player_ids['second_name']

player_ids = player_ids.drop(columns = ['first_name', 'second_name'])


player_data = pd.merge(player_ids, player_data, how = 'left', on = 'name')



In [8]:
gameweek_data = pd.merge(player_ids, gameweek_data, how = 'left', on = 'name')

In [12]:
## Points by Gameweek - trial visualization

alt.data_transformers.enable('vegafusion')

week_by_week_data = gameweek_data

## teams = sorted(gameweek_data['team'].unique())
## players = sorted(gameweek_data['name'].unique())

week_by_week_data['team_player'] = week_by_week_data['team'] + '_' + week_by_week_data['name']
options = sorted(week_by_week_data['team_player'].unique())

selection = alt.param(name = 'team_player', value = options[0],
                        bind = alt.binding_select(options=options, name = "Team-Player: "))

points_by_gameweek = (
    alt.Chart(week_by_week_data)
    .mark_line(point=alt.OverlayMarkDef(size=100))
    .encode(x = "gameweek:O", y = "total_points:Q",
            tooltip = [
                alt.Tooltip('name:N', title = 'Name'),
                alt.Tooltip('total_points:Q', title = 'Points Scored')
            ])
    .add_params(selection)
    .transform_filter
        (alt.datum['team_player'] == selection)
    .interactive()
)


points_by_gameweek

alt.Chart(...)

In [20]:

## Price by Gameweek


alt.data_transformers.enable('vegafusion')

price_by_gameweek = (
    alt.Chart(gameweek_data)
    .transform_calculate(value = 'datum.value/10') ## reflects pound value of player
    .mark_line(
        point=alt.OverlayMarkDef(size = 100)
    )
    .encode(x="gameweek:O", y="value:Q",
            tooltip=[
                alt.Tooltip('name:N', title = "Name"),
                alt.Tooltip('value:Q', title = "Price")
            ])
    .add_params(selection)
    .transform_filter(alt.datum['team_player'] == selection)
)

price_by_gameweek




alt.Chart(...)

In [18]:
## Points/price visualization

alt.data_transformers.enable('vegafusion')

points_per_pound = (
    alt.Chart(gameweek_data)
    .transform_calculate(
        value = 'datum.value/10',
    )
    .transform_calculate(
        points_per_pound = 'datum.total_points/datum.value'
    )
    .mark_line(
        point = alt.OverlayMarkDef(size = 100)
    )
    .encode(
        x = "gameweek:O", y = "points_per_pound:Q",
        tooltip= [
            alt.Tooltip('name:N', title = "Name"),
            alt.Tooltip('points_per_pound:Q', title = "Price per Pound", format = '.2f')
        ]
    )
    .add_params(selection)
    .transform_filter(alt.datum['team_player'] == selection)
)


points_per_pound


alt.Chart(...)

In [ ]:
## Expected Threat Line Chart

In [ ]:
## Histogram - bin highlights the player

In [ ]:
## Donut Chart - proportion of points scored (goals, clean sheets, defcon, etc.)

In [ ]:
## Next Opponents - strength of team